In [ ]:
# Core ML stack
!pip install -U transformers datasets accelerate peft bitsandbytes

# Optional but useful
!pip install -U sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU name: Tesla T4


In [ ]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import whoami

whoami()

{'type': 'user',
 'id': '6853976c326d003c53883052',
 'name': 'HugSena13',
 'fullname': 'Senash Hettiarachchi',
 'email': 'hettiarachchisenash@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1772323200,
 'isPro': False,
 'avatarUrl': '/avatars/c4bbe3addbf5212c729c890b80047253.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Llama model token',
   'role': 'read',
   'createdAt': '2026-02-02T01:30:45.567Z'}}}

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

# LLaMA models do NOT have a pad token by default
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
model.config.use_cache = False

In [ ]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",   # ← add
        "up_proj",     # ← add
        "down_proj"    # add
    ]
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [ ]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/chunk-rag Data/slm1_finetune_chat.jsonl"


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=DATASET_PATH,
    split="train"
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 19000
})


In [ ]:
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

Train size: 17100
Eval size: 1900


In [ ]:
# def tokenize_chat(example):
#     text = tokenizer.apply_chat_template(
#         example["messages"],
#         tokenize=False,
#         add_generation_prompt=False
#     )

#     tokenized = tokenizer(
#         text,
#         truncation=True,
#         max_length=1024,
#         padding="max_length"
#     )

#     tokenized["labels"] = tokenized["input_ids"].copy()
#     return tokenized

In [ ]:
def tokenize_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=640,
        padding="max_length"
    )

    # Build labels: mask everything except the assistant turn
    input_ids = tokenized["input_ids"]
    labels = [-100] * len(input_ids)

    # Find where the assistant response starts
    # Re-tokenize just the prompt (system + user) to find the cutoff point
    prompt_messages = example["messages"][:-1]  # all but last (assistant) message
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=640)["input_ids"]
    prompt_len = len(prompt_ids)

    # Only compute loss on assistant tokens
    for i in range(prompt_len, len(input_ids)):
        labels[i] = input_ids[i]

    tokenized["labels"] = labels
    return tokenized

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_chat,
    remove_columns=train_dataset.column_names,
    batched=False,
    load_from_cache_file=False
)

tokenized_eval = eval_dataset.map(
    tokenize_chat,
    remove_columns=eval_dataset.column_names,
    batched=False,
    load_from_cache_file=False
)

Map:   0%|          | 0/17100 [00:00<?, ? examples/s]

Map:   0%|          | 0/1900 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_train[0].keys())
print(tokenized_train[0]["labels"][-20:])

dict_keys(['input_ids', 'attention_mask', 'labels'])
[128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009]


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/slm1_lora_out",

    # Core training
    num_train_epochs=2,
    learning_rate=1e-4,
    per_device_train_batch_size=1,   # Reduced from 4 to 1 to save memory
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # Increased from 4 to 16 to keep effective batch size at 16

    # Optimizer / precision
    fp16=True,
    optim="paged_adamw_8bit",

    # Evaluation & logging (v5-compatible)
    eval_strategy="steps",   # 🔧 renamed in v5
    eval_steps=200, #was 200 changed to 50
    logging_steps=1, #was 50 changed to 1
    save_steps=200, #was 200 changed to 50
    save_total_limit=3,

    # Scheduler
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    # Trainer behavior
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    remove_unused_columns=False
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import Trainer
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
print("Trainable parameters:")
model.print_trainable_parameters()

print("\nSample batch:")
batch = next(iter(trainer.get_train_dataloader()))
for k, v in batch.items():
    print(k, v.shape)

Trainable parameters:
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

Sample batch:
input_ids torch.Size([1, 640])
attention_mask torch.Size([1, 640])
labels torch.Size([1, 640])


In [ ]:
import json

lengths = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 200:
            break
        sample = json.loads(line)
        # Concatenate all message contents as plain text and tokenize directly
        full_text = " ".join([m["content"] for m in sample["messages"]])
        tokens = tokenizer(full_text)["input_ids"]
        lengths.append(len(tokens))

print(f"Max length:   {max(lengths)}")
print(f"Mean length:  {sum(lengths)/len(lengths):.0f}")
print(f"90th pctile:  {sorted(lengths)[int(len(lengths)*0.9)]}")

Max length:   506
Mean length:  408
90th pctile:  464


In [ ]:
import json

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    first_line = f.readline()

sample = json.loads(first_line)

print("Top-level keys:", sample.keys())
print("Type of messages:", type(sample.get("messages")))
print("First message:", sample.get("messages")[0] if sample.get("messages") else "MISSING")
print("Raw line preview:", first_line[:300])

Top-level keys: dict_keys(['messages'])
Type of messages: <class 'list'>
First message: {'role': 'system', 'content': 'You are a medical constraint-aware verifier for a Neurology/Neurosurgery RAG system.\n\nYour job is STRICT binary classification.\n\nYou must output FAIL if ANY of the following are true:\n\n1) The Target Chunk explicitly contradicts a hard constraint in the query.\n2) The Target Chunk discusses a different pathology than the one in the query.\n3) The Target Chunk discusses a different anatomical region unrelated to the query.\n4) The Target Chunk is clearly unrelated to answering the question.\n5) The Target Chunk cannot reasonably help answer the query.\n\nYou must output PASS only if:\n- The chunk is applicable to the query AND\n- No hard constraint violations exist.\n\nOutput STRICT JSON in this format:\n\n{\n  "constraints": {...},\n  "reasoning": "...",\n  "label": "PASS or FAIL"\n}\n'}
Raw line preview: {"messages": [{"role": "system", "content": "You are a medi

In [ ]:
import os
from transformers import TrainerState

CHECKPOINT_DIR = "/content/drive/MyDrive/slm1_lora_out"

def get_last_checkpoint(checkpoint_dir):
    if not os.path.exists(checkpoint_dir):
        return None
    checkpoints = [
        os.path.join(checkpoint_dir, d)
        for d in os.listdir(checkpoint_dir)
        if d.startswith("checkpoint-")
    ]
    if not checkpoints:
        return None
    # Return the most recent checkpoint
    return max(checkpoints, key=lambda x: int(x.split("-")[-1]))

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    print("No checkpoint found, starting from scratch.")

trainer.train(resume_from_checkpoint=last_checkpoint)

Resuming from checkpoint: /content/drive/MyDrive/slm1_lora_out/checkpoint-2600


Step,Training Loss,Validation Loss
2800,0.478547,0.476210
3000,0.365350,0.472903
3200,0.390309,0.469281
3400,0.428104,0.467655
3600,0.418885,0.466051
3800,0.402574,0.464883
4000,0.398320,0.464535
4200,0.430182,0.464481


TrainOutput(global_step=4276, training_loss=0.17022434700806638, metrics={'train_runtime': 15832.3119, 'train_samples_per_second': 2.16, 'train_steps_per_second': 0.27, 'total_flos': 3.73373547577344e+17, 'train_loss': 0.17022434700806638, 'epoch': 2.0})

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/slm1_lora_adapter_2"

trainer.model.save_pretrained(SAVE_PATH)

In [ ]:
tokenizer.save_pretrained(SAVE_PATH)

('/content/drive/MyDrive/slm1_lora_adapter_2/tokenizer_config.json',
 '/content/drive/MyDrive/slm1_lora_adapter_2/chat_template.jinja',
 '/content/drive/MyDrive/slm1_lora_adapter_2/tokenizer.json')

In [ ]:
import os

os.listdir(SAVE_PATH)

['README.md',
 'adapter_model.safetensors',
 'adapter_config.json',
 'chat_template.jinja',
 'tokenizer_config.json',
 'tokenizer.json']

**Evaluation** **Phase**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
from peft import PeftModel

ADAPTER_PATH = "/content/drive/MyDrive/slm1_lora_adapter_2"

model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
model.print_trainable_parameters()

trainable params: 0 || all params: 3,237,063,680 || trainable%: 0.0000


In [ ]:
import json
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import time
import random

# Load the unseen data
MIRIAD_PATH = "/content/drive/MyDrive/chunk-rag Data/miriad_neurology.json"

with open(MIRIAD_PATH, "r", encoding="utf-8") as f:
    miriad_data = json.load(f)

random.seed(42)
samples = random.sample(miriad_data, 50)

# Initialize OpenAI client
client = OpenAI(api_key="OPENAI_API_KEY")

# Your exact constraint schema
CONSTRAINT_SCHEMA = {
    "Patient_Demographics": [
        "Neonatal",
        "Pediatric",
        "Adult",
        "Geriatric",
        "Pregnant",
        "N/A"
    ],
    "Acuity_Phase": [
        "Acute",
        "Chronic",
        "Pre-op",
        "Post-op",
        "N/A"
    ],
    "Pathology": "FREE_TEXT",
    "Anatomy": "FREE_TEXT",
    "Modality_Procedure": "FREE_TEXT",
    "Negative_Constraint": "FREE_TEXT"
}

# Your exact prompt template
TEACHER_PROMPT_TEMPLATE_V2 = """
You are a medical constraint and applicability verifier for a Neurology/Neurosurgery RAG system.

Your job is STRICT binary classification.

You must output FAIL if ANY of the following are true:

1) The Target Chunk explicitly contradicts a hard constraint in the query.
2) The Target Chunk discusses a different pathology than the one in the query.
3) The Target Chunk discusses a different anatomical region unrelated to the query.
4) The Target Chunk is clearly unrelated to answering the question.
5) The Target Chunk cannot reasonably help answer the query.

You must output PASS only if:
- The chunk is applicable to the query AND
- No hard constraint violations exist.

Important Rules:

- Do NOT judge answer completeness.
- Do NOT fail simply because details are missing.
- Only fail if the chunk is conflicting OR irrelevant.
- Medical topical mismatch = FAIL.
- Different condition/procedure/context = FAIL.

Constraint Schema:
- Patient_Demographics: {demographics}
- Acuity_Phase: {acuity}
- Pathology: FREE_TEXT
- Anatomy: FREE_TEXT
- Modality_Procedure: FREE_TEXT
- Negative_Constraint: FREE_TEXT

CRITICAL INSTRUCTIONS:

STEP 1:
Extract constraints ONLY from the User Query text.
Do NOT look at the Target Chunk when extracting constraints.

STEP 2:
After constraints are extracted from the User Query,
compare them against the Target Chunk.

Never copy terminology from the Target Chunk into the constraints field.
The constraints field must reflect the User Query only.

User Query:
\"\"\"{query}\"\"\"

Target Chunk:
\"\"\"{answer}\"\"\"

Output JSON ONLY:

{{
  "constraints": {{
    "Patient_Demographics": "... or N/A",
    "Acuity_Phase": "... or N/A",
    "Pathology": "... or N/A",
    "Anatomy": "... or N/A",
    "Modality_Procedure": "... or N/A",
    "Negative_Constraint": "... or N/A"
  }},
  "reasoning": "Brief explanation of why PASS or FAIL.",
  "label": "PASS or FAIL"
}}
"""

# Labeling function
def label_with_gpt4o(query, chunk):
    prompt = TEACHER_PROMPT_TEMPLATE_V2.format(
        demographics=CONSTRAINT_SCHEMA["Patient_Demographics"],
        acuity=CONSTRAINT_SCHEMA["Acuity_Phase"],
        query=query,
        answer=chunk
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=500,
            response_format={"type": "json_object"}
        )

        result = json.loads(response.choices[0].message.content)
        return result.get("label", "UNKNOWN"), result.get("constraints", {}), result.get("reasoning", "")

    except Exception as e:
        print(f"Error: {e}")
        return None, None, None

# CREATE BALANCED DATASET: 25 PASS (real pairs) + 25 FAIL (mismatched pairs)
test_cases = []

print("Creating 25 PASS cases (matched pairs)...")
# 25 PASS cases - use real matched pairs
for i in tqdm(range(25), desc="PASS cases"):
    sample = samples[i]
    label, constraints, reasoning = label_with_gpt4o(sample["question"], sample["answer"])

    if label:
        test_cases.append({
            "query": sample["question"],
            "target_chunk": sample["answer"],
            "constraints": constraints,
            "reasoning": reasoning,
            "label": label
        })
    time.sleep(0.5)

print("\nCreating 25 FAIL cases (mismatched pairs)...")
# 25 FAIL cases - create mismatches
for i in tqdm(range(25, 50), desc="FAIL cases"):
    sample = samples[i]
    # Use a chunk from a different sample (offset by 10 to ensure mismatch)
    wrong_chunk = samples[(i + 10) % 50]["answer"]

    label, constraints, reasoning = label_with_gpt4o(sample["question"], wrong_chunk)

    if label:
        test_cases.append({
            "query": sample["question"],
            "target_chunk": wrong_chunk,
            "constraints": constraints,
            "reasoning": reasoning,
            "label": label
        })
    time.sleep(0.5)

# Shuffle the dataset
validation_df = pd.DataFrame(test_cases).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nLabeled {len(validation_df)} test cases")
print("\nLabel distribution:")
print(validation_df["label"].value_counts())
validation_df.head()

Creating 25 PASS cases (matched pairs)...


PASS cases: 100%|██████████| 25/25 [01:41<00:00,  4.05s/it]



Creating 25 FAIL cases (mismatched pairs)...


FAIL cases: 100%|██████████| 25/25 [01:49<00:00,  4.38s/it]


Labeled 50 test cases

Label distribution:
label
PASS    25
FAIL    25
Name: count, dtype: int64


,query,target_chunk,constraints,reasoning,label
0,What treatment options are typically used for ...,The treatment strategies for pineal parenchyma...,"{'Patient_Demographics': 'Geriatric', 'Acuity_...",The Target Chunk discusses treatment options f...,PASS
1,How does MRgFUS differ from other methods of B...,Diffusion changes in the early phase after str...,"{'Patient_Demographics': 'N/A', 'Acuity_Phase'...",The Target Chunk discusses stroke and motor re...,FAIL
2,What are the main clinical symptoms associated...,Studies have shown that HTLV-1 DNA is detected...,"{'Patient_Demographics': 'N/A', 'Acuity_Phase'...",The Target Chunk discusses HTLV-1 and its effe...,FAIL
3,What is the role of estrogen receptors in neur...,Fibrin glues are used in neurosurgery for vari...,"{'Patient_Demographics': 'N/A', 'Acuity_Phase'...",The Target Chunk discusses neurosurgery and pr...,FAIL
4,What diagnostic tools can be used to different...,2-[18F]-fluoro-2-deoxy-D-glucose positron emis...,"{'Patient_Demographics': 'N/A', 'Acuity_Phase'...",The Target Chunk discusses FDG-PET as a diagno...,PASS


In [ ]:
# Save to Drive so you don't have to re-label if session crashes
VALIDATION_SAVE_PATH = "/content/drive/MyDrive/chunk-rag Data/validation_unseen_labeled.jsonl"

with open(VALIDATION_SAVE_PATH, "w", encoding="utf-8") as f:
    for _, row in validation_df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"Saved {len(validation_df)} validation samples to {VALIDATION_SAVE_PATH}")

Saved 50 validation samples to /content/drive/MyDrive/chunk-rag Data/validation_unseen_labeled.jsonl


In [ ]:
# import json
# import pandas as pd

# LABELED_PATH = "/content/drive/MyDrive/chunk-rag Data/slm1_training_data_v2.jsonl"

# rows = []
# with open(LABELED_PATH, "r", encoding="utf-8") as f:
#     for line in f:
#         rows.append(json.loads(line))

# df = pd.DataFrame(rows)

# print("Total labeled rows:", len(df))
# df["label"].value_counts()

Total labeled rows: 19000


,count
label,
FAIL,12419
PASS,6581


In [ ]:
# df = df.drop_duplicates(
#     subset=["query", "target_chunk", "label"]
# ).reset_index(drop=True)

# df["label"].value_counts()

,count
label,
FAIL,12419
PASS,6581


In [ ]:
# pass_samples = df[df["label"] == "PASS"].sample(n=25, random_state=42)
# fail_samples = df[df["label"] == "FAIL"].sample(n=25, random_state=42)

# validation_df = pd.concat([pass_samples, fail_samples]).sample(frac=1, random_state=42).reset_index(drop=True)

# print(validation_df["label"].value_counts())

label
PASS    25
FAIL    25
Name: count, dtype: int64


In [ ]:
validation_df.head()

,query,target_chunk,constraints,reasoning,label
0,What are the surgical management options for p...,The surgical management of PCNSL typically inv...,"{'Patient_Demographics': 'N/A', 'Acuity_Phase'...",The Target Chunk discusses the surgical manage...,PASS
1,How does the use of EM neuronavigation in neur...,Epidemiological studies suggest that four to e...,"{'Patient_Demographics': 'Pediatric', 'Acuity_...",The Target Chunk discusses convulsive status e...,FAIL
2,What are cerebral microbleeds (CMBs) and how a...,The increasing use of intracranial imaging has...,"{'Patient_Demographics': 'N/A', 'Acuity_Phase'...","The Target Chunk discusses arachnoid cysts, wh...",FAIL
3,Why are CSF biomarkers not currently recommend...,"In some instances, a combined transcranial and...","{'Patient_Demographics': 'N/A', 'Acuity_Phase'...",The Target Chunk discusses a surgical approach...,FAIL
4,What are the criteria to consider when choosin...,The criteria to consider when choosing a surgi...,"{'Patient_Demographics': 'N/A', 'Acuity_Phase'...",The Target Chunk discusses criteria for choosi...,PASS


In [ ]:
# import json
# import torch

# def get_model_prediction(row):
#     messages = [
#         {
#             "role": "system",
#             "content": "You are a medical constraint verification model."
#         },
#         {
#             "role": "user",
#             "content": f"""Query:
# {row['query']}

# Target Chunk:
# {row['target_chunk']}

# Constraints:
# {json.dumps(row['constraints'], ensure_ascii=False)}"""
#         }
#     ]

#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True
#     )

#     inputs = tokenizer(text, return_tensors="pt").to(model.device)

#     # Get the input length to properly extract only new tokens
#     input_length = inputs.input_ids.shape[1]

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=10,  # Increase from 3 to 10 for safety
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id
#         )

#     # Decode ONLY the newly generated tokens
#     generated_tokens = outputs[0][input_length:]
#     prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip().upper()

#     # Debug: print first few predictions to verify
#     # print(f"Raw prediction: '{prediction}'")

#     if "PASS" in prediction:
#         return "PASS"
#     elif "FAIL" in prediction:
#         return "FAIL"
#     else:
#         return "UNKNOWN"

In [ ]:
import json
import torch

def get_model_prediction(row):
    # Use the EXACT same system prompt as training
    messages = [
        {
            "role": "system",
            "content": """You are a medical constraint-aware verifier for a Neurology/Neurosurgery RAG system.

Your job is STRICT binary classification.

You must output FAIL if ANY of the following are true:

1) The Target Chunk explicitly contradicts a hard constraint in the query.
2) The Target Chunk discusses a different pathology than the one in the query.
3) The Target Chunk discusses a different anatomical region unrelated to the query.
4) The Target Chunk is clearly unrelated to answering the question.
5) The Target Chunk cannot reasonably help answer the query.

You must output PASS only if:
- The chunk is applicable to the query AND
- No hard constraint violations exist.

Output STRICT JSON in this format:

{
  "constraints": {...},
  "reasoning": "...",
  "label": "PASS or FAIL"
}
"""
        },
        {
            "role": "user",
            "content": f"""Query:
{row['query']}

Target Chunk:
{row['target_chunk']}"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,  # Increase to allow full JSON output
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_length:]
    prediction_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Try to parse the JSON first
    try:
        pred_json = json.loads(prediction_text)
        return pred_json.get("label", "UNKNOWN")
    except:
        # Fallback: check if PASS or FAIL appears in text
        pred_upper = prediction_text.upper()
        if "PASS" in pred_upper and "FAIL" not in pred_upper:
            return "PASS"
        elif "FAIL" in pred_upper:
            return "FAIL"
        else:
            return "UNKNOWN"

In [ ]:
from tqdm import tqdm

predictions = []

for idx, row in tqdm(validation_df.iterrows(), total=len(validation_df)):
    pred = get_model_prediction(row)
    predictions.append(pred)

validation_df["prediction"] = predictions

100%|██████████| 50/50 [12:45<00:00, 15.30s/it]


In [ ]:
accuracy = (validation_df["label"] == validation_df["prediction"]).mean()
print("Validation Accuracy:", round(accuracy * 100, 2), "%")

Validation Accuracy: 100.0 %


In [ ]:
accuracy = (validation_df["label"] == validation_df["prediction"]).mean()
print("Validation Accuracy:", round(accuracy * 100, 2), "%")

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(
    validation_df["label"],
    validation_df["prediction"],
    labels=["PASS", "FAIL"]
)

print(cm)

Validation Accuracy: 100.0 %
[[25  0]
 [ 0 25]]


In [ ]:
validation_df[
    (validation_df["label"] == "FAIL") &
    (validation_df["prediction"] == "PASS")
].head(5)

,query,target_chunk,constraints,reasoning,label,prediction
